# Understanding the Widefield + Sleep Dataset





## The four files

| # | File | What it holds | Shape / form | Format |
|---|------|---------------|--------------|--------|
| 1 | `brainMaskStruct.mat` | the spatial reference — which pixels are brain, and the Allen atlas region outlines | 2-D mask + index list + outlines | MATLAB **v7.3 (HDF5)** |
| 2 | `NREM_WF_3.mat` | the widefield imaging movie | `pixels × frames` | v7.3 or v7 |
| 3 | `09102020.SleepState.states.mat` | WAKE / NREM / REM intervals for the whole day | list of `[start, stop]` | MATLAB **v7** |
| 4 | `09102020.SlowWaves.mat` | ephys-detected slow waves (UP / DOWN states) | list of `[start, stop]` + event times | MATLAB **v7** |

## The single most important idea: one shared clock ⏱️

Everything in this dataset is expressed in **seconds of "ephys time"** — the master
clock of the electrophysiology recording.

```
ephys time (seconds) ────────────────────────────────────────────────▶
   │
   │  SleepState:   [ WAKE ][ NREM ][ REM ][ NREM ]...      (whole day)
   │  SlowWaves:            |UP|DN|UP|DN|UP|DN|...          (during NREM)
   │  Widefield:            ▓▓▓▓▓▓▓▓▓▓  (a movie, one frame every 1/100 s)
   │                        └── frame t = WF_T_START + frame_index / WF_FS
   ▼
```

- The **sleep states** and **slow waves** are already stored in seconds.
- The **widefield movie** is stored as bare frames; we reconstruct each frame's
  timestamp from a known start time and the frame rate.

Once every file speaks "seconds," we can cross-reference them freely.

> ⚠️ **Sanity check for this specific example.** The widefield file lives under a
> folder dated `09072020`, but the state files are named `09102020...`. Files from
> the *same* recording session should share the same date/basename. Before trusting
> any cross-file alignment (Section 7), **confirm these files really come from the
> same session** — otherwise the shared clock doesn't apply and the alignment is
> meaningless. If they're from different days, just point the paths at a matched set.


## 1. Imports

In [ ]:
import os
import numpy as np
import scipy.io as sio          # reads MATLAB v7 (and earlier) .mat files
import h5py                     # reads MATLAB v7.3 .mat files (they are HDF5 under the hood)
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Nicer default figure size
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['figure.dpi'] = 100

## 2. Configuration — *edit only this cell*

All paths and recording parameters live here so you never have to hunt through the
code to change them.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASEPATH   = '/Users/elisachinigo/Documents/Widefield_Data/Mice/mouse47/Conv/09072020'
WF_PATH    = os.path.join(BASEPATH, 'Processedvid3', 'NREM_WF_3.mat')
MASK_PATH  = '/Users/elisachinigo/Documents/Widefield_Data/brainMaskStruct.mat'

# The two ephys-derived files. Point these at the matching session (see the warning
# in the intro about the date mismatch).
STATE_PATH = '/Users/elisachinigo/Documents/Widefield_Data/Mice/mouse47/Conv/09072020/09102020.SleepState.states.mat'
SW_PATH    = '/Users/elisachinigo/Documents/Widefield_Data/Mice/mouse47/Conv/09072020/09102020.SlowWaves.mat'

# ── Widefield recording parameters ───────────────────────────────────────────
WF_FS      = 100.0    # Hz  — widefield frame rate
WF_T_START = 2347.0   # s   — ephys time of frame 0 in NREM_WF_3

# ── A time window to look at closely (ephys seconds) ─────────────────────────
T_START = 2483.5
T_END   = 2484.8

# ── Display settings ─────────────────────────────────────────────────────────
CLIM = (-0.2, 0.2)    # widefield colour limits (ΔF/F)

# Quick existence check so you get a friendly message instead of a cryptic error later
for label, p in [('widefield', WF_PATH), ('brain mask', MASK_PATH),
                 ('sleep states', STATE_PATH), ('slow waves', SW_PATH)]:
    print(f'{"✓" if os.path.exists(p) else "✗ MISSING"}  {label:12s} {p}')

## 3. A universal `.mat` explorer  🔍  *(learn this first!)*

Before you can *use* a file you need to know what's *inside* it. MATLAB saves data
in two very different flavours, and Python reads them with two different libraries:

| MATLAB save version | How to tell | Python reader |
|---|---|---|
| **v7 and earlier** | header starts with the text `MATLAB 5.0` | `scipy.io.loadmat` |
| **v7.3** | it's actually an **HDF5** file (magic bytes `\x89HDF`) | `h5py` |

MATLAB switches to v7.3 automatically for large variables (>2 GB), which is why the
widefield movie and the brain mask are often v7.3 while the small state files are v7.

The helpers below (a) tell you which version a file is, and (b) print its internal
structure as an indented tree. Run the explorer on **any** unfamiliar `.mat` file —
it's the fastest way to discover field names without opening MATLAB.

In [ ]:
def mat_version(path):
    """Return 'v7.3 (HDF5)' or 'v7 or earlier' by sniffing the first bytes."""
    with open(path, 'rb') as f:
        head = f.read(8)
    if head == b'\x89HDF\r\n\x1a\n':      # HDF5 magic number
        return 'v7.3 (HDF5)'
    if head[:6] == b'MATLAB':
        return 'v7 or earlier'
    return 'unknown'


def _short(x):
    """One-line description of a leaf value."""
    if isinstance(x, np.ndarray):
        return f'ndarray  shape={x.shape}  dtype={x.dtype}'
    if isinstance(x, (bytes, str)):
        s = x.decode() if isinstance(x, bytes) else x
        return f'str: {s[:50]!r}'
    return f'{type(x).__name__}: {x!r}'[:70]


def explore_scipy(obj, indent=0, max_depth=6):
    """Recursively print a scipy-loaded (simplify_cells=True) structure."""
    pad = '   ' * indent
    if indent > max_depth:
        print(pad + '...'); return
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.startswith('__'):            # skip __header__, __version__, __globals__
                continue
            if isinstance(v, (dict, list)):
                print(f'{pad}{k}/')
                explore_scipy(v, indent + 1, max_depth)
            else:
                print(f'{pad}{k}:  {_short(v)}')
    elif isinstance(obj, list):
        print(f'{pad}[list]  len={len(obj)}')
        if obj:
            explore_scipy(obj[0], indent + 1, max_depth)
    else:
        print(f'{pad}{_short(obj)}')


def explore_h5(node, name='/', indent=0, max_depth=6):
    """Recursively print an h5py (MATLAB v7.3) structure."""
    pad = '   ' * indent
    if isinstance(node, h5py.Group):
        print(f'{pad}{name}/')
        if indent >= max_depth:
            print('   ' * (indent + 1) + '...'); return
        for k in node.keys():
            if k.startswith('#'):            # skip #refs#, #subsystem# internals
                continue
            explore_h5(node[k], k, indent + 1, max_depth)
    else:  # dataset
        is_ref = h5py.check_dtype(ref=node.dtype) is not None
        tag = '  [object references → cell/struct array]' if is_ref else ''
        print(f'{pad}{name}:  shape={node.shape}  dtype={node.dtype}{tag}')


def explore(path, max_depth=6):
    """Print the version and full structure of any .mat file."""
    v = mat_version(path)
    print(f'FILE   : {os.path.basename(path)}')
    print(f'VERSION: {v}\n' + '-' * 60)
    if v == 'v7.3 (HDF5)':
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                if k.startswith('#'):
                    continue
                explore_h5(f[k], k, 0, max_depth)
    else:
        m = sio.loadmat(path, simplify_cells=True)
        explore_scipy(m, 0, max_depth)

### Try it: what version is each file?

In [ ]:
for p in [MASK_PATH, WF_PATH, STATE_PATH, SW_PATH]:
    if os.path.exists(p):
        print(f'{mat_version(p):16s}  {os.path.basename(p)}')

## 4. File 1 — `brainMaskStruct.mat` : the spatial reference

This file is the **map** that turns the widefield data (a flat list of pixels) back
into a picture of the brain, and links it to the **Allen Common Coordinate
Framework (CCF)** — the standard mouse-brain atlas.

Run the explorer to see what's inside:

In [ ]:
explore(MASK_PATH)

You should see (names may vary slightly) a `brainMaskStruct` containing:

- **`WarpData.brainMask`** — a 2-D true/false image. `True` = this pixel is inside
  the brain. Its size (e.g. `220 × 136`) is the *canvas* every widefield frame is
  painted onto.
- **`WarpData.lin_brainMask`** — a list of *linear indices*, one per brain pixel.
  This is the lookup table that says *"brain pixel #k belongs at position … in the
  2-D image."* There are exactly as many entries here as there are rows in the
  widefield movie — that's the connection between the two files.
- **`resizedDorsalMap.edgeOutline`** — the outlines of cortical regions from the
  Allen atlas (visual, somatosensory, motor cortex, …), so we can draw region
  borders on top of the movie. This is the "referencing to the Allen CCF."

### Two classic MATLAB→Python gotchas live here

1. **Column-major vs row-major order.** MATLAB numbers pixels going *down columns
   first* (Fortran order); NumPy goes *across rows first* (C order). So when we turn
   a linear index into an `(row, col)` position we must say `order='F'`.
2. **1-indexing vs 0-indexing.** MATLAB counts from 1, Python from 0, so we subtract 1.
3. **HDF5 transposes 2-D arrays** relative to MATLAB, so we `.T` the mask back.

The next cell loads everything and handles all three.

In [ ]:
def deref(ref, f):
    """Follow one HDF5 object reference and return the array it points to."""
    return f[ref][()]

with h5py.File(MASK_PATH, 'r') as f:
    # brainMask: HDF5 stores 2-D arrays transposed vs MATLAB → .T restores it
    brain_mask = f['brainMaskStruct']['WarpData']['brainMask'][()].T.astype(bool)

    # lin_brainMask: flat list of MATLAB linear indices (1-indexed) → subtract 1
    lin_flat = (f['brainMaskStruct']['WarpData']['lin_brainMask'][()]
                .ravel().astype(int) - 1)

    # edgeOutline: a cell array → an array of references we must dereference one by one
    edge_refs = f['brainMaskStruct']['resizedDorsalMap']['edgeOutline'][()].ravel()
    edges = []
    for ref in edge_refs:
        arr = deref(ref, f).T          # (N points, 2): columns are [row, col]
        if arr.size > 0:
            edges.append(arr)

nrows, ncols = brain_mask.shape

# Turn each linear index into (row, col). order='F' because MATLAB is column-major.
pix_rows, pix_cols = np.unravel_index(lin_flat, (nrows, ncols), order='F')

print(f'Canvas size    : {nrows} × {ncols} pixels')
print(f'Brain pixels   : {lin_flat.size}   (← this must equal the widefield row count)')
print(f'Region outlines: {len(edges)}')

### Visualise the mask and the Allen region outlines

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

# Left: the raw boolean mask
ax1.imshow(brain_mask, cmap='gray', origin='upper')
ax1.set_title(f'brainMask  ({nrows}×{ncols})\nwhite = brain pixel')
ax1.axis('off')

# Right: mask + Allen CCF region borders
ax2.imshow(brain_mask, cmap='gray', origin='upper', alpha=0.3)
for edge in edges:
    if edge.ndim == 2 and edge.shape[1] >= 2:
        ax2.plot(edge[:, 1], edge[:, 0], 'r', linewidth=0.8)
ax2.set_title('Allen CCF region outlines\n(edgeOutline)')
ax2.axis('off')

plt.tight_layout()
plt.show()

## 5. File 2 — `NREM_WF_3.mat` : the widefield movie

This is the imaging data: a movie of the dorsal cortex, but stored **compressed**.
Instead of `height × width × frames` (which would waste space on the ~60% of pixels
that aren't brain), it's stored as:

```
data  =  (n_brain_pixels)  ×  (n_frames)
```

Each **row** is one brain pixel's brightness over time; each **column** is one video
frame. The values are ΔF/F — fractional change in fluorescence, a proxy for neural
activity. To turn a column back into a picture we use the `lin_brainMask` lookup
from File 1.

Let's explore it first:

In [ ]:
explore(WF_PATH, max_depth=3)

### Load the movie

The loader tries `scipy` first and falls back to `h5py` for v7.3 files. Note the
`.T` in the h5py branch: h5py reads MATLAB arrays with the axes reversed, so a
`(pixels × frames)` matrix comes in as `(frames × pixels)` and we flip it back.

In [ ]:
def load_widefield(path):
    """Load the widefield data matrix as (n_pixels, n_frames), float32."""
    try:
        m = sio.loadmat(path, simplify_cells=True)
        data = np.asarray(m['NREM_widefield']['data'], dtype=np.float32)
        print('Loaded with scipy.io (v7)')
        return data, m['NREM_widefield']
    except Exception:
        pass
    with h5py.File(path, 'r') as f:
        raw = f['NREM_widefield']['data'][()]   # comes in as (frames, pixels)
        data = raw.T.astype(np.float32)         # → (pixels, frames)
        # grab any sibling fields too (e.g. a timestamps vector, if present)
        fields = {k: f['NREM_widefield'][k][()] for k in f['NREM_widefield'].keys()}
    print('Loaded with h5py (v7.3 / HDF5)')
    return data, fields


data, wf_struct = load_widefield(WF_PATH)
n_pixels, n_frames = data.shape
print(f'Widefield data : {n_pixels} pixels × {n_frames} frames')

# The pixel count MUST match the brain mask, or the reshape in the next step is wrong
assert n_pixels == lin_flat.size, (
    f'Mismatch: movie has {n_pixels} pixels but mask has {lin_flat.size}. '
    'Usually a transpose problem — check the .T in the loader.')
print('✓ pixel count matches the brain mask')

### Give every frame a timestamp (put the movie on the shared clock)

The movie file stores bare frames — no time axis. We reconstruct one. Some files
*do* save a `timestamps` field; the cell below uses it if it exists, and otherwise
rebuilds the clock from `WF_T_START` and `WF_FS`.

> **Gotcha to watch for:** the "rebuild from start + rate" approach assumes the
> frames are one continuous run. If `NREM_WF_3` were a *concatenation* of several
> separate NREM bouts, the frames would **not** be evenly spaced in ephys time and
> you'd need the real per-frame timestamps instead. The name suggests a single bout,
> so continuous is a safe default here — but now you know what to check.

In [ ]:
# Look for a saved timestamps field first
ts_field = None
if isinstance(wf_struct, dict):
    for k in wf_struct:
        if 'time' in k.lower() or 'stamp' in k.lower():
            ts_field = np.ravel(np.asarray(wf_struct[k], dtype=float))

if ts_field is not None and ts_field.size == n_frames:
    wf_t = ts_field
    print('Using saved timestamps from the file')
else:
    wf_t = np.arange(n_frames) / WF_FS + WF_T_START
    print('Reconstructed timestamps from WF_T_START + frame / WF_FS')

print(f'Movie spans {wf_t[0]:.2f} – {wf_t[-1]:.2f} s of ephys time '
      f'({(wf_t[-1]-wf_t[0]):.1f} s total)')

### Expand a compressed frame back into a picture

This is the payoff of File 1. `expand_vid` scatters each brain pixel back to its
`(row, col)` position, leaving non-brain pixels as `NaN` (so they render blank).

In [ ]:
def expand_vid(data, pix_rows, pix_cols, shape):
    """(n_pixels, n_frames) → (rows, cols, n_frames), NaN outside the brain."""
    rows, cols = shape
    vid = np.full((rows, cols, data.shape[1]), np.nan, dtype=np.float32)
    vid[pix_rows, pix_cols, :] = data
    return vid


def plot_frame(vid, frame_idx, t=None, clim=CLIM, ax=None):
    """Show one widefield frame with the brain mask and region outlines."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    frame = np.ma.masked_where(~brain_mask, vid[:, :, frame_idx])
    norm = TwoSlopeNorm(vmin=clim[0], vcenter=0, vmax=clim[1])
    im = ax.imshow(frame, cmap='RdBu_r', norm=norm, origin='upper',
                   interpolation='nearest')
    for edge in edges:
        if edge.ndim == 2 and edge.shape[1] >= 2:
            ax.plot(edge[:, 1], edge[:, 0], 'k', linewidth=0.6, alpha=0.5)
    title = f'frame {frame_idx}'
    if t is not None:
        title += f'   |   t = {t[frame_idx]:.3f} s'
    ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='ΔF/F')
    return ax


# Expand just the frames inside our window (cheaper than the whole movie)
in_window = np.where((wf_t >= T_START) & (wf_t <= T_END))[0]
vid_window = expand_vid(data[:, in_window], pix_rows, pix_cols, (nrows, ncols))
t_window = wf_t[in_window]
print(f'{len(in_window)} frames in the window {T_START}–{T_END} s')

plot_frame(vid_window, frame_idx=0, t=t_window)
plt.show()

> **Building the animated movie:** your original notebook already has a lovely
> `make_animation` function. It works the same way as `plot_frame` above, just
> updating the image each frame. (One small fix for that cell: the line
> `fig.patch.set_facecolor('none', transpare)` should be `fig.patch.set_facecolor('white')`
> — `transpare` isn't defined and `set_facecolor` takes a single colour.) Drop it in
> here once the static frame looks right.

## 6. File 3 — `SleepState.states.mat` 

This file scores the **entire day** into behavioural states: **WAKE**, **NREM**
(non-REM / slow-wave sleep), and **REM**. It follows the Buzsáki-lab *buzcode*
`.states.mat` convention, where a single struct (`SleepState`) holds the intervals.

The important new data shape here is the **interval list**: an `N × 2` array where
each row is `[start_second, stop_second]`. This is how *all* episodic data
(states, events, epochs) is stored — compact and easy to test membership against.

Explore it:

In [ ]:
explore(STATE_PATH)

Expect something like:

```
SleepState/
   ints/
      WAKEstate:  ndarray shape=(n, 2)     ← [start, stop] seconds
      NREMstate:  ndarray shape=(n, 2)
      REMstate :  ndarray shape=(n, 2)
   idx/                                     ← (optional) a sample-by-sample label vector
   detectorinfo/                            ← how/when the scoring was done
```

The exact field names can vary between pipelines (`NREMstate` vs `NREM`, etc.), so
instead of hard-coding them we **search** for whatever matches. This loader works
for both v7 and v7.3 files and returns a plain nested dictionary.

In [ ]:
def h5_to_dict(node, f):
    """Recursively convert an h5py group/dataset into nested dicts/arrays."""
    if isinstance(node, h5py.Group):
        return {k: h5_to_dict(node[k], f) for k in node.keys() if not k.startswith('#')}
    data = node[()]
    if h5py.check_dtype(ref=node.dtype) is not None:      # object references
        vals = [h5_to_dict(f[r], f) for r in np.ravel(data) if r]
        return vals[0] if len(vals) == 1 else vals
    arr = np.array(data)
    return arr.T if arr.ndim == 2 else arr               # undo HDF5 transpose


def load_struct(path, structname=None):
    """Load a MATLAB struct (v7 or v7.3) as a nested dict."""
    if mat_version(path) == 'v7.3 (HDF5)':
        with h5py.File(path, 'r') as f:
            keys = [k for k in f.keys() if not k.startswith('#')]
            name = structname if structname in keys else keys[0]
            return h5_to_dict(f[name], f)
    m = sio.loadmat(path, simplify_cells=True)
    keys = [k for k in m if not k.startswith('__')]
    name = structname if structname in keys else keys[0]
    return m[name]


def find_intervals(struct, keywords):
    """
    Pull [N x 2] interval arrays whose field name contains any of `keywords`.
    Looks inside struct['ints'] if present, else the struct itself.
    Returns {field_name: (N, 2) array}.
    """
    container = struct.get('ints', struct) if isinstance(struct, dict) else struct
    out = {}
    if isinstance(container, dict):
        for k, v in container.items():
            if any(kw.lower() in k.lower() for kw in keywords):
                a = np.atleast_2d(np.asarray(v, dtype=float))
                if a.shape[1] != 2 and a.shape[0] == 2:   # fix a stray 2xN
                    a = a.T
                if a.size:
                    out[k] = a
    return out


sleep = load_struct(STATE_PATH, 'SleepState')
states = find_intervals(sleep, ['WAKE', 'NREM', 'REM'])

print('Sleep-state intervals found:')
for name, iv in states.items():
    total = (iv[:, 1] - iv[:, 0]).sum()
    print(f'  {name:12s}: {len(iv):4d} episodes, {total/60:7.1f} min total')

### Plot the whole day at a glance

In [ ]:
def plot_hypnogram(states, ax=None, highlight=None):
    """Draw coloured bars for each state across the whole recording."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 2.2))
    colors = {'WAKE': '#e07b39', 'NREM': '#3b6ea5', 'REM': '#4caf50'}
    def color_for(name):
        for key, c in colors.items():
            if key.lower() in name.lower():
                return c
        return 'gray'
    for name, iv in states.items():
        c = color_for(name)
        ax.broken_barh([(s, e - s) for s, e in iv], (0, 1),
                       facecolors=c, label=name.replace('state', ''))
    if highlight is not None:
        ax.axvspan(highlight[0], highlight[1], color='k', alpha=0.25, zorder=5)
        ax.axvline(highlight[0], color='k', lw=0.8)
        ax.axvline(highlight[1], color='k', lw=0.8)
    ax.set_yticks([]); ax.set_xlabel('ephys time (s)')
    ax.set_title('Hypnogram — sleep state across the recording')
    ax.legend(loc='upper right', ncol=3, fontsize=8, framealpha=0.9)
    return ax

plot_hypnogram(states, highlight=(T_START, T_END))
plt.tight_layout(); plt.show()

### Ask: what state is the mouse in during our window?

A helper that tests which interval a timestamp falls into — the everyday operation
you'll do with interval data.

In [ ]:
def state_at(t, states):
    """Return the state name active at time t (seconds), or None."""
    for name, iv in states.items():
        for s, e in iv:
            if s <= t <= e:
                return name.replace('state', '')
    return None

mid = (T_START + T_END) / 2
print(f'At t = {mid:.1f} s the mouse is in: {state_at(mid, states)}')
print(f'(Our widefield file is called NREM_WF_3, so we expect NREM.)')

## 7. File 4 — `SlowWaves.mat` : UP and DOWN states

During NREM sleep the cortex oscillates between two regimes, detected from the
ephys:

- **UP state** — neurons active, depolarised, firing.
- **DOWN state** — neurons briefly go silent (hyperpolarised). In widefield ΔF/F a
  DOWN state typically shows up as a dip in activity.

This file (buzcode `.events.mat` style) stores them the same way as the sleep
states — as `[start, stop]` interval lists — usually inside `SlowWaves.ints.UP` and
`SlowWaves.ints.DOWN`, plus a `timestamps` vector marking each event.

Explore it:

In [ ]:
explore(SW_PATH)

In [ ]:
sw = load_struct(SW_PATH, 'SlowWaves')

# UP / DOWN interval lists
ud = find_intervals(sw, ['UP', 'DOWN'])
print('Slow-wave intervals found:')
for name, iv in ud.items():
    dur = (iv[:, 1] - iv[:, 0])
    print(f'  {name:8s}: {len(iv):5d} events, mean duration {dur.mean()*1000:6.1f} ms')

# event timestamps, if present
if isinstance(sw, dict) and 'timestamps' in sw:
    sw_ts = np.ravel(np.asarray(sw['timestamps'], dtype=float))
    print(f'\ntimestamps: {sw_ts.size} events, '
          f'spanning {sw_ts.min():.1f}–{sw_ts.max():.1f} s')

### Plot the slow waves inside our window

A simple "event strip": each UP/DOWN interval drawn as a coloured bar. This is how
you'd eyeball whether events line up with something else.

In [ ]:
def plot_events(intervals, window, ax=None):
    """Draw interval bars that overlap `window` = (t0, t1)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 1.8))
    t0, t1 = window
    colors = {'UP': '#c0392b', 'DOWN': '#2c3e50'}
    for name, iv in intervals.items():
        c = next((col for key, col in colors.items() if key.lower() in name.lower()), 'gray')
        segs = [(s, e - s) for s, e in iv if e >= t0 and s <= t1]
        ax.broken_barh(segs, (0, 1), facecolors=c, label=name)
    ax.set_xlim(t0, t1); ax.set_yticks([])
    ax.set_xlabel('ephys time (s)')
    ax.set_title(f'Slow waves in {t0}–{t1} s')
    ax.legend(loc='upper right', fontsize=8)
    return ax

plot_events(ud, (T_START, T_END))
plt.tight_layout(); plt.show()

## 8. Putting it together — the shared clock in action  🎯

Here's the whole point of the dataset design. In one figure we overlay, on the same
time axis:

1. the **widefield signal** (averaged over all brain pixels) — the imaging,
2. the **DOWN-state intervals** — the ephys events,

…and we already confirmed (Section 6) that this window is **NREM** sleep. If the
files are aligned, cortical activity should tend to *dip* during DOWN states.

In [ ]:
# 1. Widefield: mean ΔF/F across all brain pixels, per frame, within the window
wf_trace = data[:, in_window].mean(axis=0)

# 2. DOWN-state intervals overlapping the window
down = next((iv for name, iv in ud.items() if 'down' in name.lower()), np.empty((0, 2)))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_window, wf_trace, color='#3b6ea5', lw=1.4, label='widefield mean ΔF/F')
for i, (s, e) in enumerate(down):
    if e >= T_START and s <= T_END:
        ax.axvspan(s, e, color='0.6', alpha=0.4,
                   label='DOWN state' if i == 0 else None)
ax.set_xlim(T_START, T_END)
ax.set_xlabel('ephys time (s)'); ax.set_ylabel('mean ΔF/F')
ax.set_title(f'Widefield activity vs. ephys DOWN states  —  state: {state_at(mid, states)}')
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

print('If the shaded DOWN states line up with dips in the blue trace, the imaging')
print('and the ephys are correctly aligned on the shared clock. 🎉')

### Compare a frame during an UP state vs a DOWN state

The spatial version of the same idea — do the *pictures* differ between states?

In [ ]:
def first_frame_in(interval_list, t_axis):
    """Index of the first widefield frame that falls inside any given interval."""
    for s, e in interval_list:
        hit = np.where((t_axis >= s) & (t_axis <= e))[0]
        if hit.size:
            return hit[0]
    return None

up = next((iv for name, iv in ud.items() if name.lower().startswith('up')), np.empty((0, 2)))
i_up   = first_frame_in(up,   t_window)
i_down = first_frame_in(down, t_window)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
if i_up is not None:
    plot_frame(vid_window, i_up, t=t_window, ax=axes[0]); axes[0].set_title('during an UP state')
if i_down is not None:
    plot_frame(vid_window, i_down, t=t_window, ax=axes[1]); axes[1].set_title('during a DOWN state')
plt.tight_layout(); plt.show()

## 9. Your turn — exercises 📝

Work through these to check your understanding. All the helper functions you need
are already defined above.

1. **State budget.** Compute the fraction of the *whole recording* spent in WAKE,
   NREM, and REM. (Hint: sum interval durations from `states`, divide by the total
   span.) Make a bar chart or pie chart.

2. **DOWN-state statistics.** Plot a histogram of DOWN-state durations. What's the
   median? Does it match the ~50–300 ms range typically reported?

3. **Events within a state.** How many DOWN states fall inside NREM intervals?
   (They should be *almost all* of them — use `state_at` on each DOWN-state start
   time.) Are there any outside NREM, and what might that mean?

4. **Move the window.** Change `T_START` / `T_END` in the config cell to a *different*
   NREM period and re-run Sections 5–8. Do you still see DF/F dips during DOWN states?

5. **Region-specific trace.** Instead of averaging over *all* brain pixels in
   Section 8, average only pixels inside one Allen region (use the `edges` outlines
   to build a region mask). Does that region track DOWN states more or less clearly?


## 10. Cheat-sheet — data structures at a glance

| File | Struct → key fields | Shape | Units | Load with |
|------|--------------------|-------|-------|-----------|
| `brainMaskStruct.mat` | `WarpData.brainMask` | `rows × cols` bool | — | `h5py` |
| | `WarpData.lin_brainMask` | `(n_pixels,)` int | 1-indexed, Fortran order | `h5py` |
| | `resizedDorsalMap.edgeOutline` | cell of `(N, 2)` | pixel coords | `h5py` + `deref` |
| `NREM_WF_3.mat` | `NREM_widefield.data` | `n_pixels × n_frames` | ΔF/F | `scipy` or `h5py` (`.T`) |
| `*.SleepState.states.mat` | `SleepState.ints.{WAKE,NREM,REM}state` | `N × 2` | seconds | `scipy` |
| `*.SlowWaves.mat` | `SlowWaves.ints.{UP,DOWN}`, `.timestamps` | `N × 2`, `(M,)` | seconds | `scipy` |

**Golden rules to remember**

- Everything is on **one clock**: ephys **seconds**. The widefield movie is the only
  file that needs its timestamps *reconstructed* (`t = WF_T_START + frame / WF_FS`).
- **Dense arrays** (the movie) vs **interval lists** (`[start, stop]`, for states and
  events) are the two data shapes you'll meet constantly.
- MATLAB↔Python friction shows up as: **1-indexing** (subtract 1), **column-major
  order** (`order='F'`), and **h5py transposes** (`.T`). When a shape or a picture
  looks wrong, suspect one of these three first.
- When in doubt, **`explore(path)`** before you write any analysis code.
